# 09 · Las 10 preguntas ciegas del día 24

**Qué pregunta responde este cuaderno.** ¿El agente funciona con preguntas que nadie ha visto antes, o solo con
las 20 que nosotros mismos escribimos?

**Por qué importa.** Un sistema afinado contra su propio banco de pruebas puede estar aprendiendo a aprobar ese
examen concreto en vez de a hacer bien su trabajo. El día 24, al empezar la clase, se reparten 10 preguntas
nuevas y se ejecutan en el aula contra el repositorio ya entregado. Es la prueba de verdad.

**Qué tiene de especial.** Esas 10 preguntas llegan **sin respuesta correcta adjunta**, así que no se puede
calcular el porcentaje de aciertos como con el banco propio. Lo que sí se puede comprobar, y es lo que hace este
cuaderno, es si el agente **se comporta igual** que en casa: si tarda lo mismo, si consume lo mismo, si sigue
citando sus fuentes y si sigue reconociendo los datos que no existen.

> Este cuaderno **no inventa las preguntas**. Mientras no exista el fichero del día 24, cada celda dice
> exactamente qué le falta y no ejecuta nada.

---

*Detalle técnico.* Requisitos R10 y R15 ([01](../docs/01_requisitos_y_contratos.md)) · Guía:
[13 §8](../docs/13_skill_medicion_informe_presentacion.md) · [12 §11](../docs/12_skill_evaluadores.md).
Entrada: `ciegas.jsonl` en la raíz del repositorio; salida: `resultados/ciegas/`. El cuaderno se abre y se
ejecuta solo. Con `EJECUTAR = False` no llama a ninguna API.


In [1]:
# Arranque (igual en todos los notebooks): localiza la raíz del repo y hace importable src/
import json
import sys, pathlib
RAIZ = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ / "src"))
from IPython.display import Markdown, display
from agente10k import config
from agente10k.evaluacion import cargar_golden, evaluar

In [2]:
# Parámetros
EJECUTAR = False  # True: llama al modelo (gasta API y pide clave). False: reutiliza lo guardado en resultados/
RUTA_CIEGAS = config.RAIZ / "ciegas.jsonl"  # el fichero de las 10 preguntas ciegas del día 24
ETIQUETA_CIEGAS = "ciegas"                  # resultados/ciegas/{predicciones,puntuaciones,resumen}.jsonl|json
if EJECUTAR:
    assert config.cargar_clave(), "Falta OPENROUTER_API_KEY: copia .env.example a .env y rellénala"

## Protocolo del día 24, paso a paso

Quien ejecute esto mañana **no tiene que tocar código**. Solo esto:

1. **Traer el repositorio al día:** `git pull --ff-only` desde la carpeta del proyecto.
2. **Dejar el fichero de preguntas en su sitio:** guardar el fichero que reparta el profesor como
   `ciegas.jsonl` **en la raíz del repositorio** (la carpeta donde está `README.md`), con el mismo formato
   JSONL que `golden/golden_propio.jsonl` pero sin `respuesta_esperada` ni `cifra_esperada`.
3. **Comprobar que hay clave de API:** `OPENROUTER_API_KEY` en el fichero `.env`. Si falta, la celda de
   parámetros lo dice antes de gastar nada.
4. **Encender la ejecución:** en la celda de parámetros de arriba, cambiar `EJECUTAR = False` por
   `EJECUTAR = True`.
5. **Ejecutar el cuaderno entero** (menú *Run* → *Run All Cells*) y esperar. Cuenta con hasta 300 segundos por intento y 2 intentos por pregunta: en el peor caso unos 100 minutos para las diez
   si el proveedor va lento, y entre 6 y 15 minutos en una tarde normal.
6. **Mirar las dos salidas:** la tabla de respuestas pregunta a pregunta y el gráfico de comparación con el
   banco propio, al final del cuaderno.

**Qué se guarda solo:** `resultados/ciegas/predicciones.jsonl` (lo que respondió el agente y cómo llegó ahí),
`puntuaciones.jsonl` (lo verificable de cada respuesta) y `resumen.json` (los totales). Nada de eso pisa
mediciones anteriores.

**Si algo va mal:** si la tanda se corta a mitad, basta con volver a ejecutar la misma celda: continúa por donde
se quedó, sin repetir las preguntas ya respondidas ni volver a pagarlas.

**Qué sistema responde:** el de la entrega (`final`), es decir, búsqueda mejorada más los frenos de seguridad.
No hay que elegirlo: es el valor por defecto.


## 1. Cargar las preguntas

La celda busca `ciegas.jsonl` en la raíz del repositorio. Si está, dice cuántas preguntas ha encontrado y de qué
tipo son. Si no está, lo dice y no pasa nada más: el cuaderno no se inventa preguntas para rellenar.


In [3]:
if RUTA_CIEGAS.is_file():
    preguntas_ciegas = cargar_golden(RUTA_CIEGAS)
    conteo_familias = {}
    for p in preguntas_ciegas:
        clave = p.get("familia") or "(sin familia)"
        conteo_familias[clave] = conteo_familias.get(clave, 0) + 1
    display(Markdown(f"**{len(preguntas_ciegas)} preguntas ciegas cargadas** de `{RUTA_CIEGAS.name}` · familias: {conteo_familias}"))
else:
    preguntas_ciegas = []
    display(Markdown(
        f"⚠️ **Falta `{RUTA_CIEGAS.name}` en la raíz del repo.** Este notebook no inventa las 10 preguntas ciegas: "
        "en cuanto el fichero del día 24 se coloque ahí (mismo formato JSONL que `golden/golden_propio.jsonl`, "
        "pero sin `respuesta_esperada` ni `cifra_esperada` — docs/12 §11), esta celda las carga sola, sin tocar código."))

⚠️ **Falta `ciegas.jsonl` en la raíz del repo.** Este notebook no inventa las 10 preguntas ciegas: en cuanto el fichero del día 24 se coloque ahí (mismo formato JSONL que `golden/golden_propio.jsonl`, pero sin `respuesta_esperada` ni `cifra_esperada` — docs/12 §11), esta celda las carga sola, sin tocar código.

## 2. Responder las 10 preguntas

Aquí es donde el agente trabaja de verdad, y lo único que consume API en todo el cuaderno. Cada pregunta se
responde de cero, sin recuerdo de las anteriores.

**Qué saldrá:** una fila por pregunta con la respuesta del agente, la fuente que declara (dato oficial XBRL,
texto del informe, ambas, o ninguna cuando se abstiene), la cita concreta, el tiempo que tardó y cuánto consumió.
Como no hay respuesta correcta con la que contrastar, la columna de acierto queda vacía a propósito: **el
cuaderno no se inventa una nota**.

Si `EJECUTAR` sigue en `False` y ya existe una ejecución guardada, se reutiliza esa en lugar de volver a pagarla.


In [4]:
RUTA_RESUMEN_CIEGAS = config.RESULTADOS / ETIQUETA_CIEGAS / "resumen.json"

if EJECUTAR and preguntas_ciegas:
    puntuaciones_ciegas = evaluar(RUTA_CIEGAS, etiqueta=ETIQUETA_CIEGAS)   # sistema='final' por defecto
    display(puntuaciones_ciegas)
    resumen_ciegas = json.loads(RUTA_RESUMEN_CIEGAS.read_text(encoding="utf-8")) if RUTA_RESUMEN_CIEGAS.is_file() else {}
elif RUTA_RESUMEN_CIEGAS.is_file():
    resumen_ciegas = json.loads(RUTA_RESUMEN_CIEGAS.read_text(encoding="utf-8"))
    display(Markdown(f"Se reutiliza una ejecución ya guardada de `{ETIQUETA_CIEGAS}` "
                     f"({resumen_ciegas.get('n')} preguntas, {resumen_ciegas.get('fecha')})."))
else:
    resumen_ciegas = {}
    if not preguntas_ciegas:
        display(Markdown("`EJECUTAR=False` y sin `ciegas.jsonl`: nada que ejecutar todavía."))
    else:
        display(Markdown("`EJECUTAR=False`: no se llama al modelo. El día 24, cambia a `EJECUTAR=True` "
                         "para lanzar `evaluar()` sobre las 10 preguntas."))

`EJECUTAR=False` y sin `ciegas.jsonl`: nada que ejecutar todavía.

## 3. ¿Se comporta igual fuera de casa?

Esta es la lectura que de verdad importa, y conviene tenerla clara antes de la presentación.

**Lo que NO se puede hacer:** comparar el porcentaje de aciertos. Las preguntas ciegas no traen respuesta
correcta, así que no hay nota que comparar. Cualquiera que enseñe un «acierto en ciegas» se lo está inventando.

**Lo que SÍ se puede comparar, y es suficiente:**

| Qué se mira | Qué significaría una diferencia grande |
| --- | --- |
| Tiempo medio por pregunta | Que las preguntas nuevas le cuestan más trabajo, o que el proveedor iba peor ese día |
| Consumo medio por pregunta | Que necesita más idas y venidas para llegar a una respuesta |
| Llamadas a herramientas por pregunta | Que duda más: busca, relee, vuelve a buscar |
| Tasa de verificación (cifra que cuadra con el dato oficial, cita realmente vista) | La señal fuerte: si cae mucho fuera del banco propio, parte de la mejora medida era memoria de esas 20 preguntas, no capacidad real |
| Abstenciones | Que se encuentra con huecos y los reconoce, que es el comportamiento correcto |

**Qué saldrá:** tres gráficos de barras comparando tiempo, consumo y llamadas entre el banco propio y las
ciegas, y debajo, en una línea, las cifras de verificación de las ciegas junto al porcentaje de acierto del
banco propio — con el recordatorio de que **no son la misma métrica**.


In [5]:
import matplotlib.pyplot as plt

# Referencia del banco propio: la medición más reciente del sistema de entrega que esté completa.
# Nada se da por hecho: una carpeta a medio escribir (tanda en curso) se descarta sin romper nada.
def _resumen_completo(etiqueta):
    ruta = config.RESULTADOS / etiqueta / "resumen.json"
    if not ruta.exists():
        return None
    try:
        datos = json.loads(ruta.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError, UnicodeDecodeError):
        return None
    return datos if isinstance(datos, dict) and datos.get("n") else None


# La referencia debe estar medida con el MISMO modelo que las ciegas. Si no, el delta mezclaria
# el efecto del agente con el del proveedor y no significaria nada: es la misma regla de "mismo
# modelo en las dos filas" que aplica tabla_r11(). Antes se elegia por nombre de carpeta, lo que
# podia comparar unas ciegas de pago contra una tanda del proveedor gratuito.
MODELO_CIEGAS = (resumen_ciegas.get("modelo")
                 or config.MODELO_ID.removeprefix("openrouter:"))

_tandas_r2 = sorted((d.name for d in config.RESULTADOS.glob("final_r2_*")
                     if d.is_dir() and not d.name.endswith("_huecos")), reverse=True)
_preferencia = ["final_entregado_v2", "final_entregado", "final_pago_r2", "final_pago", "final"] + _tandas_r2 + ["candidato_07", "baseline"]
_mismo_modelo = [s for s in _preferencia
                 if (_resumen_completo(s) or {}).get("modelo") == MODELO_CIEGAS]
SISTEMA_REFERENCIA = next(iter(_mismo_modelo), None)
resumen_referencia = _resumen_completo(SISTEMA_REFERENCIA) or {}

if not resumen_ciegas:
    display(Markdown(
        f"**Pendiente:** sin `resultados/{ETIQUETA_CIEGAS}/resumen.json` no hay nada que comparar todavía. En "
        f"cuanto exista, esta celda compara sus métricas operativas contra `{SISTEMA_REFERENCIA or '(ningún sistema medido aún)'}` "
        "(golden propio, 20 preguntas) y su tasa de verificación (`sin_referencia`) frente al micro/macro del golden."))
elif not resumen_referencia:
    display(Markdown(
        f"**Pendiente:** no hay ninguna tanda del golden propio medida con `{MODELO_CIEGAS}`, que es el "
        "modelo de las ciegas. Comparar contra otro modelo mezclaría el efecto del agente con el del "
        "proveedor, así que no se dibuja nada: mide antes el golden propio con ese mismo modelo."))
else:
    metricas = [("latencia_media_s", "Tiempo medio\n(segundos por pregunta)"),
                ("tokens_medios", "Consumo medio\n(tokens por pregunta)"),
                ("llamadas_medias", "Llamadas a herramientas\n(por pregunta)")]
    fig, axes = plt.subplots(1, len(metricas), figsize=(12, 3.5))
    for ax, (clave, titulo) in zip(axes, metricas):
        valores = [resumen_referencia.get(clave), resumen_ciegas.get(clave)]
        etiquetas_x = [f"banco propio\n({SISTEMA_REFERENCIA})", "preguntas\nciegas"]
        barras = ax.bar(etiquetas_x, [v if v is not None else 0 for v in valores], color=["#888888", "#8e44ad"])
        ax.bar_label(barras, labels=[f"{v:,.1f}" if v is not None else "n/d" for v in valores], padding=3)
        ax.set_title(titulo, fontsize=10)
        ax.tick_params(axis='x', labelsize=8)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("¿Se comporta igual con preguntas nunca vistas?", fontsize=12)
    plt.tight_layout()
    plt.show()

    sr = resumen_ciegas.get("sin_referencia") or {}
    display(Markdown(
        f"**Verificación de las ciegas (sin referencia, n={sr.get('n', '—')}):** verificación global "
        f"{sr.get('verificacion', '—')} · cifra frente a XBRL {sr.get('cifra_xbrl', '—')} · cita vista "
        f"{sr.get('cita_vista', '—')} · abstenciones {sr.get('abstenciones', '—')}.\n\n"
        f"**Micro del golden propio (`{SISTEMA_REFERENCIA}`):** {resumen_referencia.get('micro')}. No es la misma "
        "métrica — las ciegas no tienen referencia — pero sirve como lectura de qué parte de la mejora es "
        "generalización y qué parte es memoria del golden."))

**Pendiente:** sin `resultados/ciegas/resumen.json` no hay nada que comparar todavía. En cuanto exista, esta celda compara sus métricas operativas contra `final_r2_t2` (golden propio, 20 preguntas) y su tasa de verificación (`sin_referencia`) frente al micro/macro del golden.

**Qué nos dice esto:** si las tres barras de la derecha se parecen a las de la izquierda, el agente se comporta
igual fuera de su banco de pruebas y la mejora medida es real. Si se disparan, o si la tasa de verificación cae
mucho, hay que decirlo en la presentación: parte de lo medido en casa sería memoria de las 20 preguntas propias,
no capacidad de generalizar.


## 4. Antes de presentar: lista de comprobación

- [ ] `ciegas.jsonl` está en la raíz y la celda 1 dice que ha cargado 10 preguntas.
- [ ] El cuaderno se ha ejecutado entero con `EJECUTAR = True` y no ha dejado ninguna celda con error.
- [ ] Existe `resultados/ciegas/resumen.json`.
- [ ] Se han mirado una por una las respuestas con `fuente = "ninguna"`: son abstenciones, y hay que saber si
      eran huecos reales del corpus (correcto) o preguntas que el agente no supo resolver (fallo).
- [ ] Se ha anotado qué preguntas agotaron el plazo de 300 segundos, para no confundir lentitud del proveedor
      con un fallo del agente.
- [ ] En la presentación se dice **explícitamente** que en las ciegas no hay porcentaje de acierto, y por qué.

**Qué contar en la presentación:** el sistema no cambió ni una línea entre el banco propio y las ciegas; lo
único que cambió fueron las preguntas. Todo lo que se mueva en las métricas operativas y en la tasa de
verificación es, por tanto, información sobre cuánto generaliza el agente.
